# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mashfiqmahi/assignment_FLyRank-AI/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content page keyed with content_id, summarized over one calander month. From the dataset what I have understood, the raw table logs one row per day per page, but since I am clustering the pages, I need per page not per day. So I will aggregate the daily rows up to one row per page for the month.


The table I will use dim_content joined with an aggregated slice of fact_content_daily_performance for the month 2026-03, a mid panel month not the final month which is s reserved as a sealed test month later. (impressions, clicks, CTR, position, engagement).



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [1]:
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [2]:
%pip -q install huggingface_hub
from huggingface_hub import whoami
print(whoami(token=os.environ["HF_TOKEN"]))

{'type': 'user', 'id': '68b6bdc7c53c1a5d8080999a', 'name': 'mashfiqmahi', 'fullname': 'Kazi Mashfiq Hossain Mahi', 'isPro': False, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/no-auth/je-I38DFmH4rBwEgPjTeT.png', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'flyrank-internship', 'role': 'fineGrained', 'createdAt': '2026-08-12T15:06:18.858Z', 'fineGrained': {'canReadGatedRepos': True, 'global': [], 'scoped': [{'entity': {'_id': '68b6bdc7c53c1a5d8080999a', 'type': 'user', 'name': 'mashfiqmahi'}, 'permissions': ['repo.content.read']}]}}}}


In [3]:
%pip -q install duckdb huggingface_hub

import os
import duckdb
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{os.environ["HF_TOKEN"]}'
    )
""")
print("Connected. HF secret registered.")

Connected. HF secret registered.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- content_id → context (grouping/joining only, not a feature)
- monthly impressions, clicks, ctr → feature (observed, known at decision time)
- gsc_avg_position → feature (observed ranking, known at decision time)
- engagement metrics (GA4) → feature, but only where ga4_data_available IS TRUE
- content metadata (e.g. days since last update) → feature (always knowable)
- trend_direction, trend_pct, is_declining_label → excluded (label-derived; would leak the answer into the clustering itself)
- No target label: clustering has no ground truth to predict, only cluster IDs to discover

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
schema_fact = con.sql("""
    DESCRIBE SELECT *
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    LIMIT 1
""").df()
print(schema_fact.to_string())

schema_dim = con.sql("""
    DESCRIBE SELECT *
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
    LIMIT 1
""").df()
print(schema_dim.to_string())

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

In [12]:
features = con.sql("""
WITH monthly AS (
    SELECT
        content_hash_id,
        SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS impressions,
        SUM(gsc_clicks)      FILTER (WHERE gsc_data_available IS TRUE) AS clicks,
        SUM(gsc_sum_position) FILTER (WHERE gsc_data_available IS TRUE) AS sum_position,
        SUM(ga4_engaged_sessions) FILTER (WHERE ga4_data_available IS TRUE) AS engaged_sessions,
        SUM(ga4_sessions)         FILTER (WHERE ga4_data_available IS TRUE) AS sessions
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY content_hash_id
)
SELECT
    m.content_hash_id,
    m.impressions,
    CASE WHEN m.impressions > 0 THEN m.clicks * 100.0 / m.impressions ELSE NULL END AS ctr_pct,
    CASE WHEN m.impressions > 0 THEN m.sum_position * 1.0 / m.impressions ELSE NULL END AS avg_position,
    CASE WHEN m.sessions > 0 THEN m.engaged_sessions * 100.0 / m.sessions ELSE NULL END AS engagement_rate_pct,
    DATE_DIFF('day', d.content_updated_date, DATE '2026-03-31') AS days_since_update
FROM monthly m
JOIN read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet') d
    ON m.content_hash_id = d.content_hash_id
""").df()
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,impressions,ctr_pct,avg_position,engagement_rate_pct,days_since_update
0,content_b7e512995f79d5a6,1140.0,0.175439,4.450877,NaN,-48
1,content_05597932fe4da067,57.0,0.000000,2.298246,NaN,-48
2,content_905aa32a0230694e,149.0,0.000000,5.637584,0.0,-48
3,content_05434271b257bb68,1421.0,0.422238,6.906404,0.0,-48
4,content_d056587ff7faca0c,2770.0,0.577617,3.950542,0.0,-97


- impressions: observed count of searches this month — already recorded by the time
  we'd review March performance, not a future value.
- ctr_pct: computed from clicks and impressions, both already observed this month.
- avg_position: observed average search ranking for the month — a measured fact, not a forecast.
- engagement_rate_pct: observed GA4 engagement this month, filtered to real (non-zero-filled)
  rows only via ga4_data_available IS TRUE.
- days_since_update: from dim_content.content_updated_date — page metadata that's always
  knowable, regardless of when we look.

In [13]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np
import pandas as pd

clean = features.dropna(subset=["impressions","ctr_pct","avg_position","engagement_rate_pct","days_since_update"])
clean = clean[clean["impressions"] > 0]
sample = clean.sample(n=min(20000, len(clean)), random_state=42)

X_honest = sample[["impressions","ctr_pct","avg_position","engagement_rate_pct","days_since_update"]].copy()
X_honest["impressions"] = np.log1p(X_honest["impressions"])
km_honest = KMeans(n_clusters=4, random_state=42, n_init=10).fit(StandardScaler().fit_transform(X_honest))
sil_honest = silhouette_score(StandardScaler().fit_transform(X_honest), km_honest.labels_)
print("Honest silhouette:", sil_honest)

# deliberate leak: a column that's just impressions*ctr restated, adds no real info
sample["clicks_leak"] = np.log1p(sample["impressions"] * sample["ctr_pct"] / 100)
X_leaky = pd.concat([X_honest, sample["clicks_leak"]], axis=1)
km_leaky = KMeans(n_clusters=4, random_state=42, n_init=10).fit(StandardScaler().fit_transform(X_leaky))
sil_leaky = silhouette_score(StandardScaler().fit_transform(X_leaky), km_leaky.labels_)
print("Leaky silhouette (with disguised duplicate column):", sil_leaky)

# delete it, keep the honest number
del sample["clicks_leak"]
print(f"Keeping honest score: {sil_honest:.3f}")

Honest silhouette: 0.3864563924148916
Leaky silhouette (with disguised duplicate column): 0.28262869076750824
Keeping honest score: 0.386


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

##### Query 1: Fact table grain: one row per (date, client, content)

In [8]:
grain_fact = con.sql("""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()
print("Fact table grain violations (should be empty):")
print(grain_fact)

# dim_content grain: one row per content item
grain_dim = con.sql("""
    SELECT content_hash_id, COUNT(*) AS c
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
    GROUP BY 1
    HAVING c > 1
    LIMIT 5
""").df()
print("dim_content grain violations (should be empty):")
print(grain_dim)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Fact table grain violations (should be empty):
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []
dim_content grain violations (should be empty):
Empty DataFrame
Columns: [content_hash_id, c]
Index: []


Grain: confirmed empty on both tables (no duplicate rows per content_hash_id, or
per report_date × client_hash_id × content_hash_id).

##### Query 2: Row count and date span

In [9]:
counts = con.sql("""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT content_hash_id) AS n_content_items,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(counts)

    n_rows  n_content_items   min_date   max_date
0  9841378           331437 2026-03-01 2026-03-31


Counts/dates: 9,841,378 rows, 331,437 unique content items, spanning exactly
2026-03-01 to 2026-03-31.

##### Query 3: — Availability, filtering with IS TRUE

In [11]:
availability = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(availability)

   total_rows  ga4_available_rows  gsc_available_rows
0     9841378              413966             3611061


Availability: 4.2% of rows have ga4_data_available IS TRUE (413,966/9,841,378);
36.7% have gsc_data_available IS TRUE (3,611,061/9,841,378).

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice has real limits, even after the contract and verification above:

1. Single-month snapshot. This contract is built and verified only on month=2026-03.
   Client portfolios, seasonality, and content mix can differ month to month, so patterns
   found here are not guaranteed to hold in other months without re-checking.

2. Uneven history and sparse GA4 coverage. Only 4.2% of March rows have usable   GA4 engagement data (ga4_data_available IS TRUE) — engagement_rate_pct is therefore built
   on a much smaller, non-random subset of the 331,437 content items than the other
   four features. This is a real limitation of this slice, not just a general warehouse
   warning.

3. GA4 rows are zero-filled, not missing, before a client's ga4_data_start. Filtering on
   ga4_data_available IS TRUE (Query 3) avoids treating those zero-fills as real zero
   engagement — but it also means my engagement features come from a smaller, non-random
   subset of rows, which could itself introduce bias I haven't measured yet.

4. No target label exists in this warehouse (trend_direction / trend_pct / is_declining_label
   are starter-CSV-only columns, not present here) — so this contract supports clustering
   only. Any future comparison against Week 1-2's label-based reasoning would need to be
   reframed, not assumed to carry over.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.